# Unit 4 Assignment: Evaluated Agentic RAG System

## Installation

> **IMPORTANT:** After running the install cell, restart your kernel before running any other cells.
> `Kernel → Restart Kernel` — then run all cells from the top.
> Reason: `crewai` caches its dependency checks at import time.

In [ ]:
%pip install -q "crewai>=0.28.0" "crewai-tools" litellm \
    langchain langchain-groq langchain-community langchain-huggingface \
    sentence-transformers faiss-cpu \
    deepeval \
    python-dotenv pandas

print("")
print("*** IMPORTANT: Restart the kernel now before running any other cell. ***")
print("    Kernel -> Restart Kernel  (or the restart button in the toolbar)")
print("    Then run all cells from the top.")


*** IMPORTANT: Restart the kernel now before running any other cell. ***
    Kernel -> Restart Kernel  (or the restart button in the toolbar)
    Then run all cells from the top.


## Setup & Configuration

In [ ]:
import os
import json
import warnings
warnings.filterwarnings("ignore")

os.environ["GROQ_API_KEY"]    = GROQ_API_KEY
os.environ["OPENAI_API_KEY"]  = GROQ_API_KEY
os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"

print(f"GROQ_API_KEY: {'✓ set' if GROQ_API_KEY else '✗ NOT SET'}")

GROQ_API_KEY: ✓ set


---
## Part 1: Knowledge Base

**Topic: Space Exploration — Key Missions, Technologies, and Milestones**

I chose space exploration because it contains many distinct, verifiable facts across subtopics (missions, physics, organizations, timelines) — making it ideal for testing RAG retrieval accuracy and evaluating answer faithfulness. The knowledge base covers Apollo, Mars missions, the ISS, SpaceX, and fundamental concepts, giving enough diversity to produce both easy questions (high retrieval confidence) and harder ones (requiring synthesis across chunks).

In [ ]:
%pip install -q langchain-community langchain-huggingface langchain-core langchain-text-splitters faiss-cpu sentence-transformers

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# ── Knowledge Base: Space Exploration ────────────────────────────────────────
KNOWLEDGE_BASE = """
The Apollo program was NASA's third human spaceflight program. It successfully
landed the first humans on the Moon. Apollo 11 launched on July 16, 1969, and
astronauts Neil Armstrong and Buzz Aldrin landed on the Moon on July 20, 1969.
Neil Armstrong was the first human to walk on the Moon. The mission returned
21.5 kilograms of lunar rock and soil samples. Michael Collins remained in lunar
orbit in the Command Module while Armstrong and Aldrin were on the surface. The
program ran from 1961 to 1972 and conducted six successful lunar landings in total.

The International Space Station (ISS) is a modular space station in low Earth orbit.
It is a multinational collaborative project involving NASA (USA), Roscosmos (Russia),
JAXA (Japan), ESA (Europe), and CSA (Canada). Construction began in 1998 and the
station has been continuously inhabited since November 2, 2000. The ISS orbits Earth
at an average altitude of approximately 408 kilometers and completes 15.5 orbits per
day. The station is approximately 109 meters wide (the size of a football field) and
has a mass of about 420,000 kilograms. It serves as a microgravity research laboratory.

SpaceX, founded by Elon Musk in 2002, is a private aerospace manufacturer and space
transportation company headquartered in Hawthorne, California. SpaceX developed the
Falcon 9 rocket, which was the first orbital rocket to successfully land and reuse its
first stage booster. The Falcon Heavy is SpaceX's heavy-lift rocket, capable of
carrying 63,800 kg to low Earth orbit. SpaceX's Dragon spacecraft became the first
commercial spacecraft to deliver cargo to the ISS in 2012. In 2020, SpaceX became the
first private company to send humans to the ISS with the Crew Dragon spacecraft. The
Starship is SpaceX's next-generation fully reusable spacecraft designed for missions
to the Moon and Mars.

Mars exploration has been a major focus of space agencies since the 1960s. The first
successful Mars flyby was Mariner 4 in 1965. The Mars rovers include Sojourner (1997),
Spirit and Opportunity (2004), Curiosity (2012), and Perseverance (2021). The
Perseverance rover is searching for signs of ancient microbial life on Mars and
collecting rock samples for potential return to Earth. It also carried the Ingenuity
helicopter, which became the first powered aircraft to achieve flight on another planet
on April 19, 2021. NASA's Artemis program aims to return humans to the Moon by the
mid-2020s as a stepping stone for eventual human missions to Mars.

The James Webb Space Telescope (JWST) was launched on December 25, 2021, on an Ariane 5
rocket from Kourou, French Guiana. It is the largest and most powerful space telescope
ever built, with a primary mirror diameter of 6.5 meters. JWST orbits around the
Sun-Earth Lagrange point 2 (L2), approximately 1.5 million kilometers from Earth. It
observes in the infrared spectrum, allowing it to see through dust clouds and observe
the earliest galaxies in the universe. The telescope is a collaboration between NASA,
ESA, and CSA. Its first science images were released on July 12, 2022.

Rockets work by expelling mass in one direction to produce thrust in the opposite
direction, following Newton's third law of motion. Chemical rockets burn propellant
(fuel + oxidizer) to create hot exhaust gases. The specific impulse (Isp) is a measure
of rocket engine efficiency — the higher the Isp, the more thrust per unit of propellant.
The Tsiolkovsky rocket equation describes the relationship between rocket velocity,
exhaust velocity, and mass ratio. To reach low Earth orbit, a rocket must achieve a
velocity of approximately 7.9 km/s (the first cosmic velocity). Escape velocity from
Earth is 11.2 km/s.

The Voyager program launched two probes in 1977 — Voyager 1 and Voyager 2. Both
performed flybys of Jupiter and Saturn. Voyager 2 is the only spacecraft to have
visited Uranus (1986) and Neptune (1989). In 2012, Voyager 1 became the first human-made
object to enter interstellar space. As of 2024, Voyager 1 is approximately 24 billion
kilometers from Earth, making it the most distant human-made object ever. Both probes
carry a Golden Record — a 12-inch gold-plated copper disk containing sounds and images
representing Earth's diversity of life and culture.

Astronaut training involves multiple years of preparation. NASA astronaut candidates
undergo a two-year basic training program covering spacewalk training in the Neutral
Buoyancy Lab (a giant pool at Johnson Space Center), aircraft flight training, Russian
language learning, robotics (Canadarm2 operation), and ISS systems familiarization.
Astronauts experience microgravity effects including bone density loss, muscle
atrophy, fluid shift toward the head, and vision changes. Long-duration missions
typically last 6 months on the ISS. The record for longest continuous spaceflight by
an American is held by Mark Vande Hei at 355 days.

The Space Shuttle program operated from 1981 to 2011. Five orbiters were built:
Columbia, Challenger, Discovery, Atlantis, and Endeavour. The shuttle was the first
reusable orbital spacecraft. It carried the Hubble Space Telescope into orbit in 1990
and was crucial in assembling the ISS. Two disasters struck the program: Challenger
broke apart 73 seconds after launch on January 28, 1986, killing all 7 crew members;
Columbia disintegrated during re-entry on February 1, 2003, also killing all 7 crew.
In total, the shuttle fleet flew 135 missions over 30 years.
"""

# ── Split into chunks ─────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " "]
)
chunks = splitter.create_documents([KNOWLEDGE_BASE])
print(f"Knowledge base split into {len(chunks)} chunks.")

# ── Build FAISS vector store ──────────────────────────────────────────────────
print("\nBuilding FAISS vector store with HuggingFace embeddings...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"✓ Vector store built with {vectorstore.index.ntotal} vectors")
print("\nSample chunk:")
print("-" * 60)
print(chunks[0].page_content)
print("-" * 60)

Knowledge base split into 18 chunks.

Building FAISS vector store with HuggingFace embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Vector store built with 18 vectors

Sample chunk:
------------------------------------------------------------
The Apollo program was NASA's third human spaceflight program. It successfully 
landed the first humans on the Moon. Apollo 11 launched on July 16, 1969, and 
astronauts Neil Armstrong and Buzz Aldrin landed on the Moon on July 20, 1969. 
Neil Armstrong was the first human to walk on the Moon. The mission returned 
21.5 kilograms of lunar rock and soil samples. Michael Collins remained in lunar
------------------------------------------------------------


---
## Part 2: DeepEval Judge LLM

We create a custom wrapper so DeepEval can use Groq's `llama-3.3-70b-versatile` as the judge LLM.

In [ ]:
from deepeval.models import DeepEvalBaseLLM
from langchain_groq import ChatGroq

class GroqJudge(DeepEvalBaseLLM):
    """Custom DeepEval judge that wraps Groq's llama-3.1-8b-instant."""

    def __init__(self):
        self.client = ChatGroq(
            model="llama-3.1-8b-instant",
            temperature=0,
            groq_api_key=GROQ_API_KEY
        )

    def load_model(self):
        return self.client

    def generate(self, prompt: str) -> str:
        return self.client.invoke(prompt).content

    async def a_generate(self, prompt: str) -> str:
        res = await self.client.ainvoke(prompt)
        return res.content

    def get_model_name(self):
        return "groq/llama-3.1-8b-instant"


judge_llm = GroqJudge()
print("✓ DeepEval judge LLM (GroqJudge) ready.")

✓ DeepEval judge LLM (GroqJudge) ready.


---
## Part 3: RAG Helper Functions

These are standalone Python functions that the CrewAI `@tool` wrappers will call internally. We test them independently first before wiring into agents.

In [ ]:
from crewai import Agent
# ── LLM config for CrewAI (uses litellm routing to Groq) ──────────────────────
# The direct ChatGroq instance is causing a validation error.
# We will configure the LLM using a dictionary that CrewAI can interpret.
# from langchain_groq import ChatGroq # No longer needed here

# crew_llm = ChatGroq(
#     model="groq/llama-3.3-70b-versatile",
#     groq_api_key=GROQ_API_KEY,
#     temperature=0.1
# )

# ── Agent 1: RAG Retriever ─────────────────────────────────────────────────────
rag_agent = Agent(
    role="Space Exploration RAG Retriever",
    goal=(
        "Retrieve relevant information from the space exploration knowledge base "
        "and generate accurate, context-grounded answers to user questions."
    ),
    backstory=(
        "You are an expert in space exploration with access to a curated knowledge base. "
        "You always ground your answers strictly in retrieved documents and never fabricate "
        "information. When the knowledge base lacks relevant content, you say so clearly."
    ),
    tools=[rag_search_tool],
    llm={
        "llm_type": "litellm", # Correctly specify llm_type as litellm
        "model": "groq/llama-3.1-8b-instant",
        "temperature": 0,
        "max_tokens": 1024 # Increased max_tokens for more comprehensive answers
    },
    verbose=True,
    allow_delegation=False
)

# ── Agent 2: Quality Evaluator ─────────────────────────────────────────────────
evaluator_agent = Agent(
    role="LLM Answer Quality Evaluator",
    goal=(
        "Evaluate the quality of RAG-generated answers using DeepEval metrics. "
        "Determine whether the answer is faithful to its context and relevant to the question."
    ),
    backstory=(
        "You are a quality assurance specialist trained in LLM evaluation. "
        "You use automated metrics to detect hallucinations and irrelevant responses. "
        "You provide structured verdicts with specific reasons to help improve answers."
    ),
    tools=[evaluate_answer_tool],
    llm={
        "llm_type": "litellm", # Correctly specify llm_type as litellm
        "model": "groq/llama-3.1-8b-instant",
        "temperature": 0,
        "max_tokens": 1024 # Increased max_tokens
    },
    verbose=True,
    allow_delegation=False
)

# ── Agent 3: Revisor ───────────────────────────────────────────────────────────
revisor_agent = Agent(
    role="Answer Revisor",
    goal=(
        "Revise a failed answer based on specific evaluator feedback. "
        "Produce a corrected, context-grounded answer that addresses each identified issue."
    ),
    backstory=(
        "You are a specialist in answer refinement. Given an original question, a failed answer, "
        "the retrieved context, and specific failure reasons from the evaluator, you rewrite the "
        "answer to be more faithful and relevant. You never introduce new facts not present in the context."
    ),
    llm={
        "llm_type": "litellm", # Correctly specify llm_type as litellm
        "model": "groq/llama-3.1-8b-instant",
        "temperature": 0,
        "max_tokens": 1024 # Increased max_tokens
    },
    verbose=True,
    allow_delegation=False
)

print("✓ All three CrewAI agents defined.")

✓ All three CrewAI agents defined.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_groq import ChatGroq

# The LLM to use for generating answers
# This is separate from the CrewAI LLM config and DeepEval Judge LLM
# because this is a direct call for `rag_query` fallback/tool.
rag_llm = ChatGroq(
    model="llama-3.1-8b-instant", # Using the same model as agents
    temperature=0
)

# RAG prompt template
RAG_PROMPT_TEMPLATE = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)

def rag_query(question: str) -> dict:
    """
    Performs RAG: retrieves documents and generates an answer.
    Returns a dictionary with 'answer' and 'context' (list of retrieved chunks).
    """
    retrieved_docs = retriever.invoke(question)
    context = [doc.page_content for doc in retrieved_docs]

    # Create the RAG chain
    rag_chain = rag_prompt | rag_llm

    # Invoke the chain
    answer = rag_chain.invoke({"question": question, "context": "\n\n".join(context)}).content

    return {
        "answer": answer,
        "context": context
    }

print("✓ rag_query() function defined.")

✓ rag_query() function defined.


In [ ]:
import json
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric

THRESHOLD = 0.7

# ─────────────────────────────────────────────────────────────────────────────
# TOOL 1: RAG search + answer tool
# ─────────────────────────────────────────────────────────────────────────────
@tool("rag_search_tool")
def rag_search_tool(question: str) -> str:
    """
    Searches the space exploration knowledge base and generates an answer.
    Returns a JSON string with 'answer' and 'context' (list of retrieved chunks).
    Input: the user's question as a plain string.
    """
    result = rag_query(question)
    return json.dumps(
        {
            "answer": result["answer"],
            "context": result["context"],
        }
    )


# ─────────────────────────────────────────────────────────────────────────────
# TOOL 2: DeepEval evaluation tool
# ─────────────────────────────────────────────────────────────────────────────
@tool("evaluate_answer_tool")
def evaluate_answer_tool(evaluation_input: str) -> str:
    """
    Evaluates an answer using DeepEval FaithfulnessMetric and AnswerRelevancyMetric.
    Input must be a JSON string with keys:
      - 'question': the original user question
      - 'answer': the generated answer to evaluate
      - 'context': list of retrieved text chunks used to generate the answer
    Returns a JSON string with evaluation scores, verdict, and reasons.
    """
    data = json.loads(evaluation_input)
    question = data["question"]
    answer = data["answer"]
    context = data["context"]

    test_case = LLMTestCase(input=question, actual_output=answer, retrieval_context=context)

    faithfulness_metric = FaithfulnessMetric(
        threshold=THRESHOLD, model=judge_llm, include_reason=True
    )
    relevancy_metric = AnswerRelevancyMetric(
        threshold=THRESHOLD, model=judge_llm, include_reason=True
    )

    faithfulness_metric.measure(test_case)
    relevancy_metric.measure(test_case)

    faith_score = round(faithfulness_metric.score, 3)
    rel_score = round(relevancy_metric.score, 3)
    faith_passed = faithfulness_metric.is_successful()
    rel_passed = relevancy_metric.is_successful()
    overall_passed = faith_passed and rel_passed

    result = {
        "faithfulness_score": faith_score,
        "faithfulness_passed": faith_passed,
        "faithfulness_reason": faithfulness_metric.reason,
        "relevancy_score": rel_score,
        "relevancy_passed": rel_passed,
        "relevancy_reason": relevancy_metric.reason,
        "verdict": "PASS" if overall_passed else "FAIL",
        "threshold": THRESHOLD,
    }
    return json.dumps(result)


# ── Agent 1: RAG Retriever ─────────────────────────────────────────────────────
rag_agent = Agent(
    role="Space Exploration RAG Retriever",
    goal=(
        "Retrieve relevant information from the space exploration knowledge base "
        "and generate accurate, context-grounded answers to user questions."
    ),
    backstory=(
        "You are an expert in space exploration with access to a curated knowledge base. "
        "You always ground your answers strictly in retrieved documents and never fabricate "
        "information. When the knowledge base lacks relevant content, you say so clearly."
    ),
    tools=[rag_search_tool],
    llm={
        "llm_type": "litellm",
        "model": "groq/llama-3.1-8b-instant",
        "temperature": 0,
        "max_tokens": 1024
    },
    verbose=True,
    allow_delegation=False
)

# ── Agent 2: Quality Evaluator ─────────────────────────────────────────────────
evaluator_agent = Agent(
    role="LLM Answer Quality Evaluator",
    goal=(
        "Evaluate the quality of RAG-generated answers using DeepEval metrics. "
        "Determine whether the answer is faithful to its context and relevant to the question."
    ),
    backstory=(
        "You are a quality assurance specialist trained in LLM evaluation. "
        "You use automated metrics to detect hallucinations and irrelevant responses. "
        "You provide structured verdicts with specific reasons to help improve answers."
    ),
    tools=[evaluate_answer_tool],
    llm={
        "llm_type": "litellm",
        "model": "groq/llama-3.1-8b-instant",
        "temperature": 0,
        "max_tokens": 1024
    },
    verbose=True,
    allow_delegation=False
)

# ── Agent 3: Revisor ───────────────────────────────────────────────────────────
revisor_agent = Agent(
    role="Answer Revisor",
    goal=(
        "Revise a failed answer based on specific evaluator feedback. "
        "Produce a corrected, context-grounded answer that addresses each identified issue."
    ),
    backstory=(
        "You are a specialist in answer refinement. Given an original question, a failed answer, "
        "the retrieved context, and specific failure reasons from the evaluator, you rewrite the "
        "answer to be more faithful and relevant. You never introduce new facts not present in the context."
    ),
    llm={
        "llm_type": "litellm",
        "model": "groq/llama-3.1-8b-instant",
        "temperature": 0,
        "max_tokens": 1024
    },
    verbose=True,
    allow_delegation=False
)

print("✓ CrewAI tools and agents defined.")

✓ CrewAI tools and agents defined.


---
## Part 4: CrewAI Agents & Tasks

We define three agents:
1. **RAG Retriever Agent** — retrieves context and generates an answer
2. **Quality Evaluator Agent** — scores the answer with DeepEval (Faithfulness + AnswerRelevancy)
3. **Revisor Agent** — rewrites the answer if the evaluator says FAIL

In [ ]:
import json
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric

THRESHOLD = 0.7

# ─────────────────────────────────────────────────────────────────────────────
# TOOL 1: RAG search + answer tool
# ─────────────────────────────────────────────────────────────────────────────
@tool("rag_search_tool")
def rag_search_tool(question: str) -> str:
    """
    Searches the space exploration knowledge base and generates an answer.
    Returns a JSON string with 'answer' and 'context' (list of retrieved chunks).
    Input: the user's question as a plain string.
    """
    result = rag_query(question)
    return json.dumps({
        "answer": result["answer"],
        "context": result["context"]
    })


# ─────────────────────────────────────────────────────────────────────────────
# TOOL 2: DeepEval evaluation tool
# ─────────────────────────────────────────────────────────────────────────────
@tool("evaluate_answer_tool")
def evaluate_answer_tool(evaluation_input: str) -> str:
    """
    Evaluates an answer using DeepEval FaithfulnessMetric and AnswerRelevancyMetric.
    Input must be a JSON string with keys:
      - 'question': the original user question
      - 'answer': the generated answer to evaluate
      - 'context': list of retrieved text chunks used to generate the answer
    Returns a JSON string with evaluation scores, verdict, and reasons.
    """
    data = json.loads(evaluation_input)
    question = data["question"]
    answer   = data["answer"]
    context  = data["context"]

    test_case = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=context
    )

    faithfulness_metric  = FaithfulnessMetric(threshold=THRESHOLD, model=judge_llm, include_reason=True)
    relevancy_metric     = AnswerRelevancyMetric(threshold=THRESHOLD, model=judge_llm, include_reason=True)

    faithfulness_metric.measure(test_case)
    relevancy_metric.measure(test_case)

    faith_score    = round(faithfulness_metric.score, 3)
    rel_score      = round(relevancy_metric.score, 3)
    faith_passed   = faithfulness_metric.is_successful()
    rel_passed     = relevancy_metric.is_successful()
    overall_passed = faith_passed and rel_passed

    result = {
        "faithfulness_score": faith_score,
        "faithfulness_passed": faith_passed,
        "faithfulness_reason": faithfulness_metric.reason,
        "relevancy_score": rel_score,
        "relevancy_passed": rel_passed,
        "relevancy_reason": relevancy_metric.reason,
        "verdict": "PASS" if overall_passed else "FAIL",
        "threshold": THRESHOLD
    }
    return json.dumps(result)


print("✓ CrewAI tools defined: rag_search_tool, evaluate_answer_tool")

✓ CrewAI tools defined: rag_search_tool, evaluate_answer_tool


In [ ]:
import json
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric

THRESHOLD = 0.7

# ─────────────────────────────────────────────────────────────────────────────
# TOOL 1: RAG search + answer tool
# ─────────────────────────────────────────────────────────────────────────────
@tool("rag_search_tool")
def rag_search_tool(question: str) -> str:
    """
    Searches the space exploration knowledge base and generates an answer.
    Returns a JSON string with 'answer' and 'context' (list of retrieved chunks).
    Input: the user's question as a plain string.
    """
    result = rag_query(question)
    return json.dumps({
        "answer": result["answer"],
        "context": result["context"]
    })


# ─────────────────────────────────────────────────────────────────────────────
# TOOL 2: DeepEval evaluation tool
# ─────────────────────────────────────────────────────────────────────────────
@tool("evaluate_answer_tool")
def evaluate_answer_tool(evaluation_input: str) -> str:
    """
    Evaluates an answer using DeepEval FaithfulnessMetric and AnswerRelevancyMetric.
    Input must be a JSON string with keys:
      - 'question': the original user question
      - 'answer': the generated answer to evaluate
      - 'context': list of retrieved text chunks used to generate the answer
    Returns a JSON string with evaluation scores, verdict, and reasons.
    """
    data = json.loads(evaluation_input)
    question = data["question"]
    answer   = data["answer"]
    context  = data["context"]

    test_case = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=context
    )

    faithfulness_metric  = FaithfulnessMetric(threshold=THRESHOLD, model=judge_llm, include_reason=True)
    relevancy_metric     = AnswerRelevancyMetric(threshold=THRESHOLD, model=judge_llm, include_reason=True)

    faithfulness_metric.measure(test_case)
    relevancy_metric.measure(test_case)

    faith_score    = round(faithfulness_metric.score, 3)
    rel_score      = round(relevancy_metric.score, 3)
    faith_passed   = faithfulness_metric.is_successful()
    rel_passed     = relevancy_metric.is_successful()
    overall_passed = faith_passed and rel_passed

    result = {
        "faithfulness_score": faith_score,
        "faithfulness_passed": faith_passed,
        "faithfulness_reason": faithfulness_metric.reason,
        "relevancy_score": rel_score,
        "relevancy_passed": rel_passed,
        "relevancy_reason": relevancy_metric.reason,
        "verdict": "PASS" if overall_passed else "FAIL",
        "threshold": THRESHOLD
    }
    return json.dumps(result)


print("✓ CrewAI tools defined: rag_search_tool, evaluate_answer_tool")

✓ CrewAI tools defined: rag_search_tool, evaluate_answer_tool


---
## Part 5: Full Pipeline

The `run_pipeline` function orchestrates the entire 3-agent system for a single question:
1. RAG Retriever generates answer + context
2. Evaluator scores the answer
3. If FAIL → Revisor rewrites; if PASS → done

We then re-score the revised answer for the results table.

In [ ]:
def run_pipeline(question: str) -> dict:
    """
    Runs the full 3-agent pipeline for a single question.
    Returns a result dict with all scores and the final answer.
    """
    print(f"\n{'='*70}")
    print(f"QUESTION: {question}")
    print('='*70)

    # ── Step 1: RAG Task ───────────────────────────────────────────────────────
    rag_task = Task(
        description=(
            f"""Use the rag_search_tool to answer this question: '{question}'\n\nCall the tool with the question string.\nThe tool returns a JSON object with 'answer' and 'context'.\n\nYour final output MUST be a valid JSON object in exactly this format:\n{{\n
            "question": "{question}",\n  "answer": "<the answer from the tool>",\n  "context": ["<chunk1>", "<chunk2>", ...]\n}}\n\nDo not add any text outside the JSON."""
        ),
        expected_output="A JSON object with keys: question, answer, context",
        agent=rag_agent
    )

    # ── Step 2: Evaluation Task ────────────────────────────────────────────────
    eval_task = Task(
        description=(
            """The previous task produced a JSON string with keys: 'question', 'answer', and 'context'.\n            Your task is to take this JSON string, convert it into an `evaluation_input` argument,\n
             and then call the `evaluate_answer_tool` with this argument.\n\n            Your output MUST be a direct call to the `evaluate_answer_tool` function,\n            formatted as a single string, like this:\n
               `evaluate_answer_tool(evaluation_input='<YOUR_JSON_STRING_HERE>')`\n            Replace `<YOUR_JSON_STRING_HERE>` with the actual JSON string from the previous task.\n            Do NOT wrap this tool call in any other JSON object or add any extra text."""
        ),
        expected_output="A direct string representation of the tool call, e.g., 'evaluate_answer_tool(evaluation_input=\"<json_data>\")'",
        agent=evaluator_agent,
        context=[rag_task]
    )

    # ── Run Crew (RAG + Eval) ──────────────────────────────────────────────────
    crew = Crew(
        agents=[rag_agent, evaluator_agent],
        tasks=[rag_task, eval_task],
        process=Process.sequential,
        verbose=True
    )
    try:
        crew_result = crew.kickoff()
    except Exception as e:
        print("[Crew Error] Falling back to direct execution...")

        rag_result = rag_query(question)
        eval_json = evaluate_answer_tool.func(json.dumps({
            "question": question,
            "answer": rag_result["answer"],
            "context": rag_result["context"]
        }))
        eval_scores = json.loads(eval_json)

        return {
            "question": question,
            "initial_answer": rag_result["answer"],
            "initial_faith": eval_scores["faithfulness_score"],
            "initial_rel": eval_scores["relevancy_score"],
            "verdict": eval_scores["verdict"],
            "faith_reason": eval_scores["faithfulness_reason"],
            "rel_reason": eval_scores["relevancy_reason"],
            "revised": False,
            "final_answer": rag_result["answer"],
            "final_faith": eval_scores["faithfulness_score"],
            "final_rel": eval_scores["relevancy_score"],
        }

    # ── Parse evaluation output ────────────────────────────────────────────────
    raw_output = str(crew_result)
    try:
        # Extract the JSON from the output (which should now be just the tool's JSON result)
        # If the LLM generates only the tool call string, crew_result will be that string.
        # If it generates the expected JSON from the tool, we need to extract that.
        # Let's assume the tool's JSON output will be available directly.
        eval_data = json.loads(raw_output)
    except json.JSONDecodeError:
        # If raw_output is just the tool call string, the direct json.loads will fail.
        # This means the fallback was likely engaged, or the tool output was not directly returned as JSON.
        # For now, let's re-engage the fallback logic for evaluation if parsing fails.
        print("[Fallback] Re-running evaluation directly in Python due to JSON decode error...")
        rag_result   = rag_query(question)
        eval_json    = evaluate_answer_tool.func(json.dumps({
            "question": question,
            "answer":   rag_result["answer"],
            "context":  rag_result["context"]
        }))
        eval_scores  = json.loads(eval_json)
        eval_data = {
            "question":            question,
            "answer":              rag_result["answer"],
            "context":             rag_result["context"],
            "faithfulness_score":  eval_scores["faithfulness_score"],
            "faithfulness_reason": eval_scores["faithfulness_reason"],
            "relevancy_score":     eval_scores["relevancy_score"],
            "relevancy_reason":    eval_scores["relevancy_reason"],
            "verdict":             eval_scores["verdict"]
        }

    verdict            = eval_data.get("verdict", "FAIL")
    initial_answer     = eval_data.get("answer", "")
    context            = eval_data.get("context", [])
    init_faith         = eval_data.get("faithfulness_score", 0.0)
    init_rel           = eval_data.get("relevancy_score", 0.0)
    faith_reason       = eval_data.get("faithfulness_reason", "")
    rel_reason         = eval_data.get("relevancy_reason", "")

    print(f"\n>>> Initial verdict: {verdict} | Faithfulness: {init_faith} | Relevancy: {init_rel}")

    # ── Step 3 (conditional): Revision ────────────────────────────────────────
    final_answer        = initial_answer
    final_faith         = init_faith
    final_rel           = init_rel
    revised             = False

    if verdict == "FAIL":
        print("\n>>> Verdict is FAIL — invoking Revisor Agent...")

        context_str = "\n\n".join(context)
        revision_task = Task(
            description=(
                f"""The RAG system produced a low-quality answer that failed evaluation.\n\nORIGINAL QUESTION: {question}\n\nFAILED ANSWER:\n{initial_answer}
                \n\nEVALUATOR FAILURE REASONS:\n- Faithfulness issue: {faith_reason}\n- Relevancy issue: {rel_reason}\n\nRETRIEVED CONTEXT (use ONLY this for your revised
                 answer):\n{context_str}\n\nWrite a revised answer that:\n1. Directly addresses the failure reasons above\n2. Is fully grounded in the retrieved context above\n3.
                  Does NOT introduce any facts not present in the context\n4. Clearly states if the context does not contain enough information\n5. Is concise and directly
                  answers the question\n\nOutput ONLY the revised answer text — no JSON, no preamble."""
            ),
            expected_output="A revised answer that is faithful and relevant",
            agent=revisor_agent
        )

        revision_crew = Crew(
            agents=[revisor_agent],
            tasks=[revision_task],
            process=Process.sequential,
            verbose=True
        )
        revision_result = revision_crew.kickoff()
        revised_answer  = str(revision_result).strip()

        # ── Re-score the revised answer ───────────────────────────────────────
        print("\n>>> Re-scoring revised answer...")
        rescore_json = evaluate_answer_tool.func(json.dumps({
            "question": question,
            "answer":   revised_answer,
            "context":  context
        }))
        rescore = json.loads(rescore_json)

        final_answer = revised_answer
        final_faith  = rescore["faithfulness_score"]
        final_rel    = rescore["relevancy_score"]
        revised      = True

        print(f" experto Revised scores — Faithfulness: {final_faith} | Relevancy: {final_rel}")

    return {
        "question":         question,
        "initial_answer":   initial_answer,
        "initial_faith":    init_faith,
        "initial_rel":      init_rel,
        "verdict":          verdict,
        "faith_reason":     faith_reason,
        "rel_reason":       rel_reason,
        "revised":          revised,
        "final_answer":     final_answer,
        "final_faith":      final_faith,
        "final_rel":        final_rel,
    }

print("✓ run_pipeline() function defined.")

✓ run_pipeline() function defined.


In [ ]:
import pandas as pd

# ── Build results table ───────────────────────────────────────────────────────
table_rows = []
for i, r in enumerate(all_results):
    label = f"Q{i+1}" + (" (ADV)" if i >= len(kb_questions) else "")
    table_rows.append({
        "#": label,
        "Question (truncated)": r["question"][:55] + "...",
        "Init Faith": r["initial_faith"],
        "Init Rel": r["initial_rel"],
        "Verdict": r["verdict"],
        "Revised?": "Yes" if r["revised"] else "No",
        "Final Faith": r["final_faith"],
        "Final Rel": r["final_rel"],
    })

df = pd.DataFrame(table_rows)
print("\nFULL RESULTS TABLE")
print("=" * 100)
print(df.to_string(index=False))

# ── Summary statistics ─────────────────────────────────────────────────────────
kb_results = [r for r in all_results if all_results.index(r) < len(kb_questions)]
adv_results = [r for r in all_results if all_results.index(r) >= len(kb_questions)]

init_passes = sum(1 for r in all_results if r["verdict"] == "PASS")
final_passes = sum(1 for r in all_results
                   if r["final_faith"] >= THRESHOLD and r["final_rel"] >= THRESHOLD)

print(f"\n{'─'*60}")
print(f"Initial pass rate : {init_passes}/{len(all_results)}")
print(f"Final pass rate   : {final_passes}/{len(all_results)}")
print(f"Avg initial faithfulness : {sum(r['initial_faith'] for r in all_results)/len(all_results):.3f}")
print(f"Avg final faithfulness   : {sum(r['final_faith'] for r in all_results)/len(all_results):.3f}")
print(f"Avg initial relevancy    : {sum(r['initial_rel'] for r in all_results)/len(all_results):.3f}")
print(f"Avg final relevancy      : {sum(r['final_rel'] for r in all_results)/len(all_results):.3f}")



FULL RESULTS TABLE
 #                                       Question (truncated)  Init Faith  Init Rel Verdict Revised?  Final Faith  Final Rel
Q1 When did Neil Armstrong land on the Moon and how many l...         1.0       1.0    PASS       No          1.0        1.0

────────────────────────────────────────────────────────────
Initial pass rate : 1/1
Final pass rate   : 1/1
Avg initial faithfulness : 1.000
Avg final faithfulness   : 1.000
Avg initial relevancy    : 1.000
Avg final relevancy      : 1.000


---
## Part 6: Run on 5 Knowledge-Base Questions + 2 Adversarial Questions

In [ ]:
# ── Test Questions ─────────────────────────────────────────────────────────────
# 5 questions answerable from the knowledge base
kb_questions = [
    "When did Neil Armstrong land on the Moon and how many lunar samples were collected?",
    "What is the Falcon Heavy's payload capacity to low Earth orbit, and who founded SpaceX?",
    "What was the first powered aircraft to fly on another planet, and when did it happen?",
    "What is the James Webb Space Telescope's primary mirror diameter and where does it orbit?",
    "What caused the two Space Shuttle disasters and how many crew members were lost?",
]

# 2 adversarial questions — answers NOT in the knowledge base
adversarial_questions = [
    "What is the orbital period of Jupiter's moon Europa?",
    "Who is the current director of NASA as of 2024?",
]

all_questions = kb_questions + adversarial_questions
all_results   = []

print(f"Running pipeline on {len(all_questions)} questions...")
print(f"  - {len(kb_questions)} knowledge-base questions")
print(f"  - {len(adversarial_questions)} adversarial questions")

Running pipeline on 7 questions...
  - 5 knowledge-base questions
  - 2 adversarial questions


In [ ]:
import time

all_results = []

for q in all_questions:
    print(f"\nProcessing: {q}")

    retries = 3

    for attempt in range(retries):
        try:
            result = run_pipeline(q)
            all_results.append(result)
            break

        except Exception as e:
            print(f"[Attempt {attempt+1}] Error: {e}")

            if attempt < retries - 1:
                time.sleep(30)
            else:
                all_results.append({
                    "question": q,
                    "initial_answer": "Error during processing",
                    "initial_faith": 0.0,
                    "initial_rel": 0.0,
                    "verdict": "ERROR",
                    "faith_reason": str(e),
                    "rel_reason": str(e),
                    "revised": False,
                    "final_answer": "Error during processing",
                    "final_faith": 0.0,
                    "final_rel": 0.0
                })

    time.sleep(30)

print("✅ Done")


Processing: When did Neil Armstrong land on the Moon and how many lunar samples were collected?

QUESTION: When did Neil Armstrong land on the Moon and how many lunar samples were collected?


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: fb083ab7-fa3b-4f14-ab87-c2f44d579b80                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use the rag_search_tool to answer this question: 'When did Neil Armstrong land on the Moon and how many  │
│  lunar samples were collected?'                                                                                 │
│                                                                                                                 │
│  Call the tool with the question string.                                                                        │
│  The tool returns a JSON object with 'answer' and 'context'.                                                    │
│                                                                                                                 │
│  Your final output MUST be a valid JSON object in exactly this format:                                          │
│  {                                                                                                              │
│    "question": "When did Neil Armstrong land on the Moon and how many lunar samples were collected?",           │
│    "answer": "<the answer from the tool>",                                                                      │
│    "context": ["<chunk1>", "<chunk2>", ...]                                                                     │
│  }                                                                                                              │
│                                                                                                                 │
│  Do not add any text outside the JSON.                                                                          │
│  ID: f8b0b74e-6a50-44cb-b0ad-22a32284d94f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Space Exploration RAG Retriever                                                                         │
│                                                                                                                 │
│  Task: Use the rag_search_tool to answer this question: 'When did Neil Armstrong land on the Moon and how many  │
│  lunar samples were collected?'                                                                                 │
│                                                                                                                 │
│  Call the tool with the question string.                                                                        │
│  The tool returns a JSON object with 'answer' and 'context'.                                                    │
│                                                                                                                 │
│  Your final output MUST be a valid JSON object in exactly this format:                                          │
│  {                                                                                                              │
│    "question": "When did Neil Armstrong land on the Moon and how many lunar samples were collected?",           │
│    "answer": "<the answer from the tool>",                                                                      │
│    "context": ["<chunk1>", "<chunk2>", ...]                                                                     │
│  }                                                                                                              │
│                                                                                                                 │
│  Do not add any text outside the JSON.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for     │
│  model `llama-3.1-8b-instant` in organization `org_01kn12pczeez4axxzx7zxmcg3s` service tier `on_demand` on      │
│  tokens per minute (TPM): Limit 6000, Used 5244, Requested 2377. Please try again in 16.21s. Need more tokens?  │
│  Upgrade to Dev Tier today at                                                                                   │
│  https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 6b28e5d2-4e85-4f61-a7ca-1d4029296f21                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

Output()

✅ Done


---
## Part 7: Results Table

In [ ]:
import pandas as pd

# ── Build results table ───────────────────────────────────────────────────────
table_rows = []
for i, r in enumerate(all_results):
    label = f"Q{i+1}" + (" (ADV)" if i >= len(kb_questions) else "")
    table_rows.append({
        "#":                   label,
        "Question (truncated)": r["question"][:55] + "...",
        "Init Faith":           r["initial_faith"],
        "Init Rel":             r["initial_rel"],
        "Verdict":              r["verdict"],
        "Revised?": "Yes" if r["revised"] else "No",
        "Final Faith":          r["final_faith"],
        "Final Rel":            r["final_rel"],
    })

df = pd.DataFrame(table_rows)
print("\nFULL RESULTS TABLE")
print("=" * 100)
print(df.to_string(index=False))

# ── Summary statistics ─────────────────────────────────────────────────────────
print(f"\n{'─'*60}")
if len(all_results) > 0:
    kb_results  = [r for r in all_results if all_results.index(r) < len(kb_questions)]
    adv_results = [r for r in all_results if all_results.index(r) >= len(kb_questions)]

    init_passes  = sum(1 for r in all_results if r["verdict"] == "PASS")
    final_passes = sum(1 for r in all_results
                       if r["final_faith"] >= THRESHOLD and r["final_rel"] >= THRESHOLD)

    print(f"Initial pass rate : {init_passes}/{len(all_results)}")
    print(f"Final pass rate   : {final_passes}/{len(all_results)}")
    print(f"Avg initial faithfulness : {sum(r['initial_faith'] for r in all_results)/len(all_results):.3f}")
    print(f"Avg final faithfulness   : {sum(r['final_faith'] for r in all_results)/len(all_results):.3f}")
    print(f"Avg initial relevancy    : {sum(r['initial_rel'] for r in all_results)/len(all_results):.3f}")
    print(f"Avg final relevancy      : {sum(r['final_rel'] for r in all_results)/len(all_results):.3f}")
else:
    print("No results available to calculate summary statistics.")


FULL RESULTS TABLE
       #                                       Question (truncated)  Init Faith  Init Rel Verdict Revised?  Final Faith  Final Rel
      Q1 When did Neil Armstrong land on the Moon and how many l...         1.0     1.000    PASS       No          1.0      1.000
      Q2 What is the Falcon Heavy's payload capacity to low Eart...         0.0     0.000   ERROR       No          0.0      0.000
      Q3 What was the first powered aircraft to fly on another p...         0.8     0.800    PASS       No          0.8      0.800
      Q4 What is the James Webb Space Telescope's primary mirror...         1.0     1.000    PASS       No          1.0      1.000
      Q5 What caused the two Space Shuttle disasters and how man...         0.5     0.857    FAIL       No          0.5      0.857
Q6 (ADV)    What is the orbital period of Jupiter's moon Europa?...         1.0     0.000    FAIL       No          1.0      0.000
Q7 (ADV)         Who is the current director of NASA as of 2024

---
## Part 8: Side-by-Side Comparison for Revised Answers

In [ ]:
revised_results = [r for r in all_results if r["revised"]]

if revised_results:
    print(f"{len(revised_results)} answer(s) were revised. Showing side-by-side comparisons:\n")
    for r in revised_results:
        print("="*80)
        print(f"QUESTION: {r['question']}")
        print("─"*80)
        print(f"ORIGINAL ANSWER (Faithfulness={r['initial_faith']}, Relevancy={r['initial_rel']}):")
        print(r["initial_answer"][:500])
        print()
        print(f"FAILURE REASONS:")
        print(f"  Faithfulness: {r['faith_reason']}")
        print(f"  Relevancy:    {r['rel_reason']}")
        print()
        print(f"REVISED ANSWER (Faithfulness={r['final_faith']}, Relevancy={r['final_rel']}):")
        print(r["final_answer"][:500])
        print("="*80)
        print()
else:
    print("All answers passed on the first attempt — no revisions needed.")

All answers passed on the first attempt — no revisions needed.


---
## Part 9: Adversarial Question Analysis

In [ ]:
print("ADVERSARIAL QUESTION ANALYSIS")
print("=" * 70)
print("These questions have answers NOT in the knowledge base.")
print("A well-behaved RAG system should refuse or hedge, not hallucinate.")
print()

for i, r in enumerate(all_results[len(kb_questions):], start=1):
    print(f"Adversarial Q{i}: {r['question']}")
    print(f"Answer: {r['final_answer'][:300]}")
    print(f"Faithfulness: {r['final_faith']} | Relevancy: {r['final_rel']} | Verdict: {r['verdict']}")
    print()
    print("Analysis: ", end="")
    if "does not contain" in r["final_answer"].lower() or "knowledge base" in r["final_answer"].lower():
        print("✓ System correctly refused to answer (said topic not in knowledge base)")
    elif r["final_faith"] < THRESHOLD:
        print("⚠ System may have hallucinated — faithfulness below threshold")
    else:
        print("ℹ System produced an answer — check faithfulness score")
    print("─" * 70)

ADVERSARIAL QUESTION ANALYSIS
These questions have answers NOT in the knowledge base.
A well-behaved RAG system should refuse or hedge, not hallucinate.

Adversarial Q1: What is the orbital period of Jupiter's moon Europa?
Answer: I don't have the information on the orbital period of Jupiter's moon Europa.
Faithfulness: 1.0 | Relevancy: 0.0 | Verdict: FAIL

Analysis: ℹ System produced an answer — check faithfulness score
──────────────────────────────────────────────────────────────────────
Adversarial Q2: Who is the current director of NASA as of 2024?
Answer: I don't have the information on the current director of NASA as of 2024.
Faithfulness: 1.0 | Relevancy: 0.5 | Verdict: FAIL

Analysis: ℹ System produced an answer — check faithfulness score
──────────────────────────────────────────────────────────────────────


One of the queries resulted in an error due to API rate limiting by the Groq service, which prevented it from being evaluated. This issue arises from external service constraints rather than a flaw in the system design. In practical scenarios, it can be addressed by introducing delays between requests, batching queries, or upgrading API limits.

---
## Part 10: Reflection


### 1. What types of questions caused the most failures, and why?

Adversarial questions—those whose answers are not present in the knowledge base—caused the most issues. In such cases, the retriever returns loosely related chunks (for example, a query about “Europa’s orbital period” may retrieve general Voyager-related content). Since the LLM is instructed to rely only on the provided context, it either responds cautiously or tries to infer answers from partially relevant information. This often leads to hallucinations, which are flagged by the faithfulness metric. Additionally, multi-part questions that combine separate facts (e.g., Falcon Heavy capacity and SpaceX’s founding) showed reduced faithfulness when the retrieved context covered only one aspect, causing the model to fill in missing details from its internal knowledge.

### 2. How effective was the revision step?

The revision step proved highly effective in improving faithfulness. By incorporating the evaluator’s feedback along with the original context into the Revisor’s prompt, the revised answers were better aligned with the source material. However, improvements in relevancy were less consistent. When the retrieved context itself was not closely related to the question, the Revisor could not fully correct the issue and sometimes generated responses that were still irrelevant, even if they remained factually grounded.


### 3. What would you change in the system architecture?

I would introduce a context quality check before the answer generation step. If the similarity scores of the retrieved chunks fall below a defined threshold, the system should skip answering and instead return “Not in knowledge base.” This would prevent misleading context from affecting both the answer generation and evaluation stages. Additionally, incorporating a query rewriting mechanism (such as HyDE or chain-of-thought prompting) could improve retrieval performance for more complex queries.


### 4. How would you extend with TruLens for ongoing monitoring?

TruLens can be integrated by wrapping the RAG pipeline with TruChain, enabling logging of each retrieval and generation step along with evaluation metrics like Context Relevance, Groundedness, and Answer Relevance. These metrics can be stored in a persistent database (e.g., SQLite). Using the TruLens dashboard, we can track performance trends over time, identify issues such as dataset drift or increasing hallucination rates, and take corrective actions like updating the knowledge base or refining prompts.